# GeoDamage AI -- final pipeline (paper-aligned)

<a href="https://colab.research.google.com/github/REPLACE_ME/Gaza-Damage-Project/blob/paper-aligned-pipeline/notebooks/02_geodamage_ai_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook implements the methodology described in Sections 3-4 of
*"GeoDamage AI: A weakly supervised six-channel U-Net pipeline for
building-level damage screening in Al-Shati Camp, Gaza"*: UNOSAT + Microsoft
building-footprint weak labels with an explicit ignore class, a compact
six-channel early-fusion U-Net, validation-only threshold calibration, and a
final building-level inventory with GIS export.

It is backed by the tested `geodamage/` package in this repo (see
`tests/test_geodamage.py` -- 9/9 passing locally, including real-data-backed
tests against the downloaded imagery/footprints/UNOSAT data), so the data
loading, weak-label construction, model architecture, loss, calibration, and
inventory/export logic have all actually been run and checked, not just
written. The one thing that has **not** been run end-to-end is full model
*training* -- that needs a GPU, so it's meant to run here on Colab.

**Read this before trusting any numbers this notebook produces:**
- The paper's exact reported numbers (pixel threshold 0.23, F1 0.982,
  building thresholds 0.570/0.918/0.955, Q20=0.055273, inventory of 2,986
  buildings, etc.) are empirical outputs of the *authors'* training run on
  their *own* manually-curated 15-section split. This notebook reproduces the
  **procedure**, not their exact numbers -- running it produces its own
  results on its own data split.
- Test (Sections 4, 5) and validation (Section 10) sections are fixed by the
  paper's text. The 12 training sections are **our own choice** (lowest
  section indices not already used), not a reconstruction of the paper's
  original overlap-audited selection (see `geodamage/config.py`).
- The "Al-Shati Camp" study-area boundary isn't mapped as a polygon in
  OpenStreetMap (only a point node exists, confirmed by direct Nominatim
  query) -- the final inventory therefore uses the union of the test
  sections' bboxes as the study area, which is **much larger** than the
  paper's tight camp boundary, so the final building count will not match
  2,986.


In [ ]:
# --- Setup: clone this repo (for Colab) and install dependencies ---
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "REPLACE_ME_WITH_GITHUB_URL_AFTER_PUSHING"  # fill in once this branch is pushed

if IN_COLAB:
    if not os.path.exists("Gaza-Damage-Project"):
        subprocess.run(["git", "clone", "-b", "paper-aligned-pipeline", REPO_URL, "Gaza-Damage-Project"], check=True)
    os.chdir("Gaza-Damage-Project")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Running locally/Jupyter from within the repo already -- just make sure
    # the repo root is on sys.path so `import geodamage` works.
    sys.path.insert(0, os.path.abspath(".."))

import geodamage  # noqa: E402  (import after path/deps setup above)
print("geodamage package OK, modules:", geodamage.__all__)


In [ ]:
# --- Download data (see data/README.md for what each source is and why) ---
# Safe to re-run: scripts skip/overwrite deterministically and total download
# size is small (~190MB) because we only pull the 15 sections actually used,
# not the full ~6GB Kaggle archive -- see scripts/download_imagery.py.
import subprocess, sys

for script in ["download_imagery.py", "download_building_footprints.py", "download_unosat.py"]:
    print(f"=== running scripts/{script} ===")
    subprocess.run([sys.executable, f"scripts/{script}"], check=True)


## 1. Spatial design (paper Sec. 3.1)

Fifteen sections, standardized to EPSG:32636 at ~8m resolution: 12 training,
1 validation (Section 10, used only for threshold calibration), 2 test
(Sections 4, 5 -- the paper's "Al-Shati test", geospatially disjoint from
train/val). See `geodamage/config.py` for the exact section lists and the
caveat about the training-section selection.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from geodamage import config, geo_utils, data_sources

sections = data_sources.list_downloaded_sections()
print("Downloaded sections:", sections)

fig, ax = plt.subplots(figsize=(8, 8))
colors = {"train": "#a6cee3", "val": "#33a02c", "test": "#e31a1c"}
for sec in sections:
    img = geo_utils.load_section_image(sec, "before")
    minx, miny, maxx, maxy = img.bbox_wgs84
    role = "test" if sec in config.TEST_SECTIONS else ("val" if sec in config.VAL_SECTIONS else "train")
    ax.add_patch(mpatches.Rectangle((minx, miny), maxx - minx, maxy - miny,
                                     facecolor=colors[role], edgecolor="black", alpha=0.5))
    ax.text((minx + maxx) / 2, (miny + maxy) / 2, str(sec), ha="center", va="center", fontsize=8)
ax.autoscale()
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Spatial split (our selection) -- analogous to paper Fig. 1")
ax.legend(handles=[mpatches.Patch(color=c, label=k) for k, c in colors.items()])
plt.show()


## 2. Weak-label construction (paper Sec. 3.2)

Asymmetric policy: UNOSAT-matched buildings -> damaged (positive).
Buildings that are far from any damage reference, have valid imagery, *and*
have a low before/after change statistic (below a Q20 percentile calibrated
on the **training pool only**) -> likely-intact. Everything else -> ignore
(excluded from loss and metrics). See `geodamage/weak_labels.py`.


In [ ]:
from geodamage import data_sources, weak_labels
import numpy as np

unosat = data_sources.load_unosat_damage_sites()
print(f"UNOSAT damage points: {len(unosat)}")

# --- Pass 1: compute the training pool's change statistic (no Q20 yet) ---
train_pool_change_stats = []
train_section_cache = {}  # section -> (before_img, after_img, buildings)
for sec in config.TRAIN_SECTIONS:
    before = geo_utils.load_section_image(sec, "before")
    after = geo_utils.load_section_image(sec, "after")
    h, w = before.image.shape[:2]
    buildings = data_sources.load_building_footprints(bbox_wgs84=before.bbox_wgs84)
    labeled = weak_labels.assign_weak_labels(buildings, before.image, after.image, before.bbox_wgs84, w, h, unosat)
    train_pool_change_stats.extend(labeled["change_stat"].dropna().tolist())
    train_section_cache[sec] = (before, after, buildings)
    print(f"  section_{sec}: {len(buildings)} buildings processed")

Q20 = weak_labels.calibrate_q20(train_pool_change_stats)
print(f"\nCalibrated Q20 (our training pool, {len(train_pool_change_stats)} buildings): {Q20:.6f}")
print(f"(paper's own reported value, from their data: 0.055273 -- not expected to match exactly)")


In [ ]:
# --- Pass 2: assign final weak labels (now with Q20 frozen) for every section,
#     and rasterize per-section pixel-level label masks for training/eval ---
section_data = {}  # section -> dict(before, after, labeled_buildings, label_mask)

for sec in config.ALL_SECTIONS:
    if sec in train_section_cache:
        before, after, buildings = train_section_cache[sec]
    else:
        before = geo_utils.load_section_image(sec, "before")
        after = geo_utils.load_section_image(sec, "after")
        buildings = data_sources.load_building_footprints(bbox_wgs84=before.bbox_wgs84)
    h, w = before.image.shape[:2]

    labeled = weak_labels.assign_weak_labels(
        buildings, before.image, after.image, before.bbox_wgs84, w, h, unosat, q20_threshold=Q20,
    )
    label_mask = geo_utils.rasterize_buildings(labeled, before.bbox_wgs84, w, h, value_field="weak_label")

    section_data[sec] = {"before": before, "after": after, "buildings": labeled, "label_mask": label_mask}
    vc = labeled["weak_label"].value_counts().to_dict()
    print(f"section_{sec}: {vc}")


## 3. Patches, model, and loss (paper Sec. 3.3-3.4)

128x128x6 non-overlapping patches (Before-RGB + After-RGB, normalized to
[0,1]); the compact early-fusion U-Net reconstructed in `geodamage/model.py`
(verified locally to land within 0.1% of the paper's reported 3,315,345
parameters); masked weighted BCE + Dice loss with the minority-class weight
computed from our own training masks (`geodamage/losses.py`).


In [ ]:
import torch
from torch.utils.data import DataLoader
from geodamage import dataset, model, losses

def to_6ch(sec):
    return dataset.stack_before_after(section_data[sec]["before"].image, section_data[sec]["after"].image)

train_pairs = [(to_6ch(s), section_data[s]["label_mask"]) for s in config.TRAIN_SECTIONS]
val_pairs = [(to_6ch(s), section_data[s]["label_mask"]) for s in config.VAL_SECTIONS]
test_pairs = [(to_6ch(s), section_data[s]["label_mask"]) for s in config.TEST_SECTIONS]

train_ds = dataset.SectionPatchDataset(train_pairs)
val_ds = dataset.SectionPatchDataset(val_pairs)
test_ds = dataset.SectionPatchDataset(test_pairs)
print(f"train patches: {len(train_ds)}, val patches: {len(val_ds)}, test patches: {len(test_ds)}")

train_loader = DataLoader(train_ds, batch_size=config.TRAIN_BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=config.TRAIN_BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}" + ("" if device.type == "cuda" else "  (!! no GPU -- training below will be very slow)"))

net = model.SixChannelUNet().to(device)
n_params = sum(p.numel() for p in net.parameters())
print(f"model parameters: {n_params:,} (paper reports {config.PAPER_REPORTED_PARAM_COUNT:,})")

train_masks = [m for _, m in train_pairs]
weight_intact = losses.compute_class_weight(train_masks)
print(f"computed likely-intact class weight: {weight_intact:.4f} (paper reports ~2.0969 from their data)")
loss_fn = losses.MaskedWeightedBCEDiceLoss(weight_damaged=1.0, weight_intact=weight_intact)


## 4. Training (paper Sec. 3.4)

Adam(lr=3e-4), batch size 8, up to 40 epochs, early stopping +
`ReduceLROnPlateau`, checkpoint on best validation loss. **This cell needs a
GPU to finish in reasonable time** -- it has not been executed as part of
building this notebook (no GPU was available in the environment used to write
and test the rest of the pipeline). Everything upstream of this cell *has*
been run end-to-end against real data.


In [ ]:
import torch.optim as optim

optimizer = optim.Adam(net.parameters(), lr=config.TRAIN_LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

best_val_loss = float("inf")
patience, patience_counter = 7, 0
checkpoint_path = config.DATA_OUTPUTS / "six_channel_unet_best.pt"
config.DATA_OUTPUTS.mkdir(parents=True, exist_ok=True)
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, config.TRAIN_MAX_EPOCHS + 1):
    net.train()
    train_losses = []
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = loss_fn(net(x), y)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    net.eval()
    val_losses = []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            val_losses.append(loss_fn(net(x), y).item())

    train_loss, val_loss = sum(train_losses) / len(train_losses), sum(val_losses) / len(val_losses)
    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    scheduler.step(val_loss)
    print(f"epoch {epoch:02d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss, patience_counter = val_loss, 0
        torch.save(net.state_dict(), checkpoint_path)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch} (best val_loss={best_val_loss:.4f})")
            break

plt.plot(history["train_loss"], label="train"); plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("masked weighted BCE + Dice loss"); plt.legend(); plt.title("Training history")
plt.show()


## 5. Pixel threshold calibration (paper Sec. 3.5)

Sweep candidate thresholds against full-section inference on the **validation
section only**, select by max macro-F1 (our explicit, documented selection
rule -- the paper doesn't fully spell out its own), then freeze before any
test-section inference.


In [ ]:
net.load_state_dict(torch.load(checkpoint_path, map_location=device))
net.eval()

def predict_section_probs(sec):
    img6ch = to_6ch(sec)
    h, w = img6ch.shape[:2]
    probs = np.zeros((h, w), dtype=np.float32)
    counts = np.zeros((h, w), dtype=np.float32)
    patch_size = config.PATCH_SIZE
    with torch.no_grad():
        for row in range(0, h, patch_size):
            for col in range(0, w, patch_size):
                r1, c1 = min(row + patch_size, h), min(col + patch_size, w)
                patch = np.zeros((patch_size, patch_size, 6), dtype=np.float32)
                patch[:r1 - row, :c1 - col] = img6ch[row:r1, col:c1]
                x = torch.from_numpy(patch).permute(2, 0, 1).unsqueeze(0).float().to(device)
                p = torch.sigmoid(net(x))[0, 0].cpu().numpy()
                probs[row:r1, col:c1] = p[:r1 - row, :c1 - col]
                counts[row:r1, col:c1] = 1
    return probs

val_probs = {sec: predict_section_probs(sec) for sec in config.VAL_SECTIONS}
val_labels_concat = np.concatenate([section_data[s]["label_mask"].ravel() for s in config.VAL_SECTIONS])
val_probs_concat = np.concatenate([val_probs[s].ravel() for s in config.VAL_SECTIONS])

from geodamage import calibration
sweep_results, best = calibration.sweep_pixel_threshold(val_probs_concat, val_labels_concat)

thresholds = [r.threshold for r in sweep_results]
plt.plot(thresholds, [r.f1 for r in sweep_results], label="damaged F1")
plt.plot(thresholds, [r.macro_f1 for r in sweep_results], label="macro F1")
plt.plot(thresholds, [r.balanced_accuracy for r in sweep_results], label="balanced accuracy")
plt.axvline(best.threshold, color="k", linestyle="--", label=f"selected={best.threshold:.2f}")
plt.xlabel("probability threshold"); plt.legend(); plt.title("Validation pixel-threshold sweep")
plt.show()

PIXEL_THRESHOLD = best.threshold  # frozen from here on
print(f"Frozen pixel threshold: {PIXEL_THRESHOLD:.3f}  (paper's frozen threshold: 0.23)")


## 6. Test evaluation (paper Sec. 4.2)

Run once, on the frozen checkpoint + frozen pixel threshold, over the
held-out test sections (4, 5) -- never touched before this cell.


In [ ]:
test_probs = {sec: predict_section_probs(sec) for sec in config.TEST_SECTIONS}
test_labels_concat = np.concatenate([section_data[s]["label_mask"].ravel() for s in config.TEST_SECTIONS])
test_probs_concat = np.concatenate([test_probs[s].ravel() for s in config.TEST_SECTIONS])

test_metrics = calibration.compute_pixel_metrics(test_probs_concat, test_labels_concat, PIXEL_THRESHOLD)
print("Pooled spatial test metrics (Sections 4+5), analogous to paper Table 2:")
for field in test_metrics.__dataclass_fields__:
    print(f"  {field}: {getattr(test_metrics, field):.4f}")


## 7. Building-level calibration and final inventory (paper Sec. 3.5, 4.3-4.5)

Building-level AI damage score = mean pixel probability within the footprint
(Eq. 1), calibrated on **validation** buildings to get a screening threshold
plus 90%/95%-specificity thresholds, then applied to buildings in the
Al-Shati study area (see the notebook-level caveat at the top about why our
study area and building count differ from the paper's).


In [ ]:
from geodamage import inventory

# Building-level scores on the validation section, for threshold calibration
val_buildings = section_data[config.VAL_SECTIONS[0]]["buildings"]
val_building_preds = [inventory.SectionPrediction(
    config.VAL_SECTIONS[0], section_data[config.VAL_SECTIONS[0]]["before"].bbox_wgs84,
    *section_data[config.VAL_SECTIONS[0]]["before"].image.shape[1::-1], val_probs[config.VAL_SECTIONS[0]],
)]
val_scored = inventory.score_buildings(val_buildings, val_building_preds)
calibratable = val_scored[val_scored["weak_label"] != config.LABEL_IGNORE]
building_thresholds = calibration.calibrate_building_thresholds(
    calibratable["ai_damage_score"].to_numpy(), calibratable["weak_label"].to_numpy(),
)
print("Building-level thresholds (ours vs paper's 0.570 / 0.918 / 0.955):", building_thresholds)


In [ ]:
# Final inventory over the Al-Shati study area (union of test-section bboxes --
# see top-of-notebook caveat), scored from the test-section predictions.
al_shati_area = inventory.al_shati_area_from_sections()
minx, miny, maxx, maxy = al_shati_area.total_bounds
al_shati_buildings = data_sources.load_building_footprints(bbox_wgs84=(minx, miny, maxx, maxy))
al_shati_buildings = al_shati_buildings[al_shati_buildings.intersects(al_shati_area.geometry.iloc[0])].reset_index(drop=True)
print(f"Buildings in Al-Shati study area: {len(al_shati_buildings)} (paper: 2,986 -- see caveat above)")

test_section_preds = [
    inventory.SectionPrediction(
        sec, section_data[sec]["before"].bbox_wgs84,
        *section_data[sec]["before"].image.shape[1::-1], test_probs[sec],
    )
    for sec in config.TEST_SECTIONS
]
final_inventory = inventory.build_inventory(al_shati_buildings, test_section_preds, building_thresholds, unosat_gdf=unosat)
print(inventory.tier_summary(final_inventory))

gpkg_path, geojson_path = inventory.export_inventory(final_inventory)
print(f"Exported: {gpkg_path}\n          {geojson_path}")


## 8. UNOSAT audit (paper Sec. 4.5)

Framed as an audit/review layer, not ground truth -- UNOSAT's own product
description explicitly calls it preliminary and non-field-validated.


In [ ]:
audited = final_inventory.dropna(subset=["ai_damage_score"])
matched = audited["unosat_matched"].sum()
print(f"Buildings with a matched UNOSAT reference: {matched} / {len(audited)}")

high_conf_tiers = {config.TIER_VERY_HIGH_CONF, config.TIER_HIGH_CONF}
review_queue = audited[audited["ai_tier"].isin(high_conf_tiers) & ~audited["unosat_matched"]]
print(f"High/very-high-confidence AI candidates without a UNOSAT label (review queue): {len(review_queue)}")


## Next step

`app/streamlit_app.py` loads `data/outputs/al_shati_building_inventory.gpkg`
(produced above) plus the trained checkpoint to serve the interactive map +
live-inference web app described in the paper's Sec. 4.4-4.5. See
`notebooks/03_web_app_demo.ipynb` for a Colab-friendly launcher.
